# Fifth CuPy example: combining operations with fuse

We begin with the same sum $c_i=a_i+b_i$, input values, vector length, and single precision as in the preceding examples. This time a Python function decorated with `cp.fuse()` describes the computation.

A single addition already requires only one elementwise operation. To demonstrate why fusion can be useful, we then compare ordinary and fused evaluations of $c_i=2a_i+3b_i$, the expression introduced in the fourth example. Both results are checked against NumPy before optional timing.

This completes the five-example CuPy sequence. It is **not** a direct counterpart of the fifth PyCUDA example: explicit CUDA kernels combined with GPU arrays were already covered in the first two CuPy notebooks. Here we introduce an additional CuPy interface.

Run the cells in order in a Colab GPU runtime. Results and timing printouts appear directly in cells; no result files are exported.

## 1. Prepare Google Colab

Select **Runtime → Change runtime type → GPU**, then connect. The commands run on the remote Linux machine. `nvidia-smi` checks GPU availability; its CUDA version describes driver support. `nvcc --version` reports the installed toolkit compiler.

We do not invoke `nvcc` to compile a user-written kernel here. The toolkit version is useful for selecting a CuPy package if none is installed. CuPy still requires working CUDA components to execute and, when needed, generate its GPU kernels.

In [1]:
!nvidia-smi
!nvcc --version

Sat Oct 17 14:29:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Use the existing CuPy installation when available. The next cell installs a CUDA-specific binary package **only if CuPy is absent**, selecting CUDA 12 or 13 from the installed toolkit. It does not install PyCUDA or a notebook extension. Do not install multiple CuPy packages in the same environment.

If an existing CuPy installation fails to import, use a fresh Colab GPU runtime before proceeding. For other environments, follow the [CuPy installation guide](https://docs.cupy.dev/en/stable/install.html).

In [2]:
import importlib.util
import re
import subprocess
import sys

if importlib.util.find_spec("cupy") is None:
    toolkit_info = subprocess.check_output(
        ["nvcc", "--version"], text=True
    )
    match = re.search(r"release\s+(\d+)\.", toolkit_info)
    if match is None or int(match.group(1)) not in (12, 13):
        raise RuntimeError("Use a Colab GPU runtime with CUDA 12 or 13, or follow the CuPy installation guide.")
    package = f"cupy-cuda{match.group(1)}x"
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    print("Installed:", package)
else:
    print("Using the existing CuPy installation.")

Using the existing CuPy installation.


## 2. Initialize CuPy

`cp` is the conventional alias for CuPy. We use device 0 throughout this standalone notebook. Unlike the PyCUDA example, there is no `pycuda.autoinit` import. CUDA failures are reported as Python exceptions.

In [3]:
import numpy as np
import cupy as cp

if cp.cuda.runtime.getDeviceCount() < 1:
    raise RuntimeError("No CUDA GPU is available. Select a GPU runtime in Colab.")
cp.cuda.Device(0).use()
properties = cp.cuda.runtime.getDeviceProperties(0)
device_name = properties["name"]
if isinstance(device_name, bytes):
    device_name = device_name.decode()
print("GPU:", device_name)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("CuPy:", cp.__version__)
print("CUDA runtime version:", cp.cuda.runtime.runtimeGetVersion())

GPU: Tesla T4
Python: 3.13.15
NumPy: 2.1.3
CuPy: 14.0.1
CUDA runtime version: 13000


## 3. Choose the vector length

Use $N=100003$ as before. CuPy manages the iteration range and launch configuration; no grid or block dimensions are specified. The limit below keeps the NumPy input-index construction within its chosen `int32` range.

In [4]:
N = 100003
assert 1 <= N <= np.iinfo(np.int32).max
print(f"Elements: {N}")
print("Launch configuration: managed by CuPy.")

Elements: 100003
Launch configuration: managed by CuPy.


## 4. Prepare the NumPy inputs

The deterministic inputs match the previous examples: $a_i=0.25(i\bmod101)$ and $b_i=0.50(i\bmod37)$. Both arrays are contiguous `float32` arrays, matching the kernel's `float` pointers.

In [5]:
indices = np.arange(N, dtype=np.int32)
h_a = np.float32(0.25) * (indices % 101).astype(np.float32)
h_b = np.float32(0.50) * (indices % 37).astype(np.float32)
assert h_a.dtype == h_b.dtype == np.float32
assert h_a.flags.c_contiguous and h_b.flags.c_contiguous
print(f"Bytes per vector: {h_a.nbytes}")

Bytes per vector: 400012


## 5. Create device arrays and transfer the inputs

For these NumPy inputs, `cp.asarray` creates device arrays and copies their values to the GPU. Their storage remains available for later GPU operations. Applying `cp.asarray` to an already compatible CuPy array need not copy it.

Unlike the first two examples, we do not allocate an output with `cp.empty_like`: the addition expression will create it. The printout below inspects metadata only, without retrieving array values. Transfers and subsequent work use the same current stream; keep the host inputs unchanged while their copies are pending.

In [6]:
d_a = cp.asarray(h_a)
d_b = cp.asarray(h_b)
print("Device array class:", type(d_a).__name__)
print("Input shape:", d_a.shape)
print("Input dtype:", d_a.dtype)
print("Device bytes per input:", d_a.nbytes)
print("Device:", d_a.device.id)

Device array class: ndarray
Input shape: (100003,)
Input dtype: float32
Device bytes per input: 400012
Device: 0


## 6. Define and call a fused Python function

A decorator changes how a function is handled. `@cp.fuse()` asks CuPy to generate a kernel for supported operations expressed in the function. We write Python array arithmetic, not a CUDA C++ source string.

With CuPy inputs, the function executes GPU work and returns a CuPy array. Its first call can generate and compile a kernel; subsequent compatible calls reuse cached code. Define the decorated function once and reuse it.

For this single addition, there are no intermediate operations to combine. This first step introduces the syntax and preserves the common numerical example; it does not promise a benefit over `d_a + d_b`.

In [7]:
def ordinary_add(a, b):
    return a + b

@cp.fuse()
def fused_add(a, b):
    return a + b

d_c = fused_add(d_a, d_b)
assert isinstance(d_c, cp.ndarray)
assert d_c.shape == (N,) and d_c.dtype == np.float32

## 7. Retrieve and validate every output

`cp.asnumpy` explicitly retrieves the GPU result as a NumPy array. Its default blocking behavior makes that result available to the CPU before returning. We use the same independent NumPy reference and tolerances as the preceding examples and check every element, including the last entries.

In [8]:
h_c = cp.asnumpy(d_c)
h_reference = h_a + h_b
if not np.all(np.isfinite(h_c)):
    raise AssertionError("The GPU result contains nonfinite values.")
np.testing.assert_allclose(h_c, h_reference, rtol=1e-6, atol=1e-6)
max_abs_error = float(np.max(np.abs(h_c - h_reference)))
print(f"Validation passed for all {N} elements.")
print(f"Maximum absolute error: {max_abs_error:.3e}")

Validation passed for all 100003 elements.
Maximum absolute error: 0.000e+00


## 8. Display the computed sum

The `c[i] (GPU)` column contains the result actually retrieved from the device. The first 12 and last 3 entries are displayed alongside the NumPy reference, with the same layout as the previous examples.

In [9]:
shown_indices = sorted(set(range(min(12, N))) | set(range(max(0, N - 3), N)))
print(f"{'i':>8} {'a[i]':>12} {'b[i]':>12} {'c[i] (GPU)':>15} {'NumPy':>12}")
for i in shown_indices:
    print(f'{i:8d} {h_a[i]:12.6f} {h_b[i]:12.6f} '
          f'{h_c[i]:15.6f} {h_reference[i]:12.6f}')

       i         a[i]         b[i]      c[i] (GPU)        NumPy
       0     0.000000     0.000000        0.000000     0.000000
       1     0.250000     0.500000        0.750000     0.750000
       2     0.500000     1.000000        1.500000     1.500000
       3     0.750000     1.500000        2.250000     2.250000
       4     1.000000     2.000000        3.000000     3.000000
       5     1.250000     2.500000        3.750000     3.750000
       6     1.500000     3.000000        4.500000     4.500000
       7     1.750000     3.500000        5.250000     5.250000
       8     2.000000     4.000000        6.000000     6.000000
       9     2.250000     4.500000        6.750000     6.750000
      10     2.500000     5.000000        7.500000     7.500000
      11     2.750000     5.500000        8.250000     8.250000
  100000     2.500000    13.000000       15.500000    15.500000
  100001     2.750000    13.500000       16.250000    16.250000
  100002     3.000000    14.000000      

## 9. Combine several operations: a weighted sum

Now evaluate $c_i=\alpha a_i+\beta b_i$ with $\alpha=2$ and $\beta=3$. Ordinary CuPy evaluation computes two array products and then adds them. These operations normally launch separate elementwise kernels and form intermediate device arrays. A memory pool can reuse their storage, but that does not eliminate the operations or the intermediate writes and reads.

The fused function expresses the same arithmetic in Python. For this supported elementwise expression, CuPy combines the operations into a generated kernel so that intermediate element values need not be stored in separate full-size arrays. The output is still a new device array.

The two functions have the same arguments and return the same mathematical result. Coefficients are `np.float32` to preserve single precision. We check both outputs against NumPy using tolerances, since arithmetic contraction/order can affect rounding for general inputs.

In [10]:
def ordinary_linear(a, b, alpha, beta):
    return alpha * a + beta * b

@cp.fuse()
def fused_linear(a, b, alpha, beta):
    return alpha * a + beta * b

alpha = np.float32(2.0)
beta = np.float32(3.0)
d_linear_ordinary = ordinary_linear(d_a, d_b, alpha, beta)
d_linear_fused = fused_linear(d_a, d_b, alpha, beta)
h_linear_ordinary = cp.asnumpy(d_linear_ordinary)
h_linear_fused = cp.asnumpy(d_linear_fused)
h_linear_reference = alpha * h_a + beta * h_b

for name, result in [("ordinary", h_linear_ordinary), ("fused", h_linear_fused)]:
    assert result.shape == h_linear_reference.shape and result.dtype == np.float32
    if not np.all(np.isfinite(result)):
        raise AssertionError(f"Nonfinite output in {name} version.")
    np.testing.assert_allclose(result, h_linear_reference, rtol=1e-6, atol=1e-6)
    print(f"{name}: validated all {N} elements; max error = "
          f"{np.max(np.abs(result-h_linear_reference)):.3e}")

print(f"{'i':>8} {'Ordinary GPU':>15} {'Fused GPU':>15} {'NumPy 2a+3b':>15}")
for i in shown_indices:
    print(f"{i:8d} {h_linear_ordinary[i]:15.6f} "
          f"{h_linear_fused[i]:15.6f} {h_linear_reference[i]:15.6f}")

ordinary: validated all 100003 elements; max error = 0.000e+00
fused: validated all 100003 elements; max error = 0.000e+00
       i    Ordinary GPU       Fused GPU     NumPy 2a+3b
       0        0.000000        0.000000        0.000000
       1        2.000000        2.000000        2.000000
       2        4.000000        4.000000        4.000000
       3        6.000000        6.000000        6.000000
       4        8.000000        8.000000        8.000000
       5       10.000000       10.000000       10.000000
       6       12.000000       12.000000       12.000000
       7       14.000000       14.000000       14.000000
       8       16.000000       16.000000       16.000000
       9       18.000000       18.000000       18.000000
      10       20.000000       20.000000       20.000000
      11       22.000000       22.000000       22.000000
  100000       44.000000       44.000000       44.000000
  100001       46.000000       46.000000       46.000000
  100002       48.0000

## 10. Optional: compare ordinary and fused expressions

Both versions use the same device inputs and create a new output on each call. A host timer measures repeated calls followed by a wait for the current stream. This includes Python overhead, output/intermediate array management, allocation or reuse from the memory pool, kernel submissions, execution, and the final wait. Input transfers, warm-up compilation, result retrieval, and validation are excluded.

This scope follows the third CuPy notebook, not the preallocated-output CUDA-event timing in the fourth. Three warm-up calls precede measurement of each function. Seven batches of 100 calls are timed; the order of the two alternatives alternates to reduce a fixed ordering bias. The table reports median batch-mean time and interquartile range (IQR).

The ratio is ordinary/fused: values above one favor fusion **for this measured scope**. Small differences may reflect variability. No benefit is guaranteed, especially for the sum alone. A timing difference does not identify kernel counts or DRAM traffic; those require profiling. Repeated use of the arrays and the allocator also affects measurements.

In [11]:
from time import perf_counter

REPEATS = 100
BATCHES = 7
assert REPEATS > 0 and BATCHES > 0
stream = cp.cuda.get_current_stream()

def check_result(device_result, reference):
    assert device_result.shape == reference.shape
    assert device_result.dtype == reference.dtype
    host_result = cp.asnumpy(device_result)
    if not np.all(np.isfinite(host_result)):
        raise AssertionError("Nonfinite GPU output.")
    np.testing.assert_allclose(host_result, reference, rtol=1e-6, atol=1e-6)

def compare_pair(title, functions, args, reference):
    # Validate and warm both implementations before measuring either.
    for function in functions.values():
        for _ in range(3):
            result = function(*args)
        stream.synchronize()
        check_result(result, reference)
        del result
    samples = {name: [] for name in functions}
    names = list(functions)
    for batch in range(BATCHES):
        order = names if batch % 2 == 0 else names[::-1]
        for name in order:
            function = functions[name]
            # Keep an output alive before timing in both alternatives,
            # so replacement/release has the same lifetime pattern.
            result = function(*args)
            stream.synchronize()
            start_time = perf_counter()
            for _ in range(REPEATS):
                result = function(*args)
            stream.synchronize()
            milliseconds = (perf_counter() - start_time) * 1000.0 / REPEATS
            samples[name].append(milliseconds)
            check_result(result, reference)  # outside timing
            del result
    print("\n" + title)
    print(f"{'Method':<14} {'Median ms/call':>16} {'IQR ms/call':>16}")
    medians = {}
    for name, values in samples.items():
        median = float(np.median(values))
        q25, q75 = np.percentile(values, [25, 75])
        medians[name] = median
        print(f"{name:<14} {median:16.6f} {q75-q25:16.6f}")
    ratio = medians["ordinary"] / medians["fused"]
    print(f"Ordinary/fused median-time ratio: {ratio:.3f}")
    print("All measured outputs validated. Transfers and compilation excluded.")
    return samples

sum_samples = compare_pair(
    "Simple sum: one operation, no intermediate products",
    {"ordinary": ordinary_add, "fused": fused_add},
    (d_a, d_b), h_reference,
)
linear_samples = compare_pair(
    "Linear combination: two products followed by a sum",
    {"ordinary": ordinary_linear, "fused": fused_linear},
    (d_a, d_b, alpha, beta), h_linear_reference,
)


Simple sum: one operation, no intermediate products
Method           Median ms/call      IQR ms/call
ordinary               0.021371         0.002478
fused                  0.024370         0.003604
Ordinary/fused median-time ratio: 0.877
All measured outputs validated. Transfers and compilation excluded.

Linear combination: two products followed by a sum
Method           Median ms/call      IQR ms/call
ordinary               0.092445         0.022061
fused                  0.029277         0.022694
Ordinary/fused median-time ratio: 3.158
All measured outputs validated. Transfers and compilation excluded.


## 11. Release the array references

After the required GPU work has completed, delete the device arrays used by the examples. CuPy's memory pool may retain the released storage for reuse. Rerun from the input-transfer cell before computing again.

In [12]:
cp.cuda.get_current_stream().synchronize()
del d_a, d_b, d_c, d_linear_ordinary, d_linear_fused
print("Device array references released.")

Device array references released.


## What does this fifth interface add?

- `RawKernel` and `RawModule` use explicit CUDA source and launch configurations.
- Ordinary array arithmetic uses existing CuPy operations.
- `ElementwiseKernel` takes a scalar CUDA C/C++ expression and generates its indexing.
- `fuse` describes supported combinations through a decorated Python function.

For the weighted sum, ordinary evaluation forms arrays for the intermediate products, whereas fusion can keep intermediate element values within the generated kernel. This can reduce launches and intermediate global-memory traffic, but does not guarantee a shorter measured runtime.

Fusion supports a subset of operations. It is not a general Python-to-CUDA compiler and does not automatically fuse arbitrary library routines, CPU code, or separate functions in the whole application. Both methods still require transfers when NumPy inputs or CPU-readable outputs are needed. Do not confuse kernel fusion with an individual fused multiply-add machine instruction.

**Preparation status:** delivered with empty outputs. Python syntax, notebook structure, unchanged inputs, and the numerical bodies of the sum and weighted-sum functions were checked on CPU. CUDA compilation and GPU execution were not available during preparation. Run all cells on Colab to obtain actual results and timings; no GPU timings are supplied here.

References: [CuPy kernel fusion](https://docs.cupy.dev/en/stable/user_guide/kernel.html#kernel-fusion), [cupy.fuse](https://docs.cupy.dev/en/stable/reference/generated/cupy.fuse.html).